# Register a cleaned dataset

The round trip: read a dataset, fix it here, and hand it back so the
platform reports it **Ready to train** — without a file leaving the machine.


In [ ]:
import orinth
import polars as pl

SOURCE = next(d for d in orinth.datasets.list() if d.task_type == "text_classification")
df = orinth.datasets.load(SOURCE.id, "train")
print(SOURCE.id, df.shape)
df.head()


## Clean

Drop empties and duplicates — the two things that make a dataset look
larger than it is.


In [ ]:
cleaned = (
    df.filter(pl.col("text").str.strip_chars().str.len_chars() > 0)
      .unique(subset=["text"])
      .select(["text", "label"])
)
print(f"{df.height} -> {cleaned.height} rows")


## Register

`register()` writes nothing itself: it uploads the rows and hands the prep
agent a plan that states the task, format, labels and split, so the result
goes through the same apply path a browser upload does.


In [ ]:
ref = orinth.datasets.register(
    cleaned,
    name=f"{SOURCE.name} (cleaned)",
    task_type="text_classification",
    columns={"text": "text", "label": "label"},
)

print(ref.id)
print(ref.splits)
print(ref.readiness.state, "|", ref.readiness.summary)
